In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import display
import ipywidgets as widgets

%matplotlib inline

In [ ]:
# Number of points
N = 200

# Generate angles
t = np.linspace(0, 2*np.pi, N, endpoint=False)

# Generate circle points
x = np.cos(t)
y = np.sin(t)

# Display the curve
plt.figure(figsize=(6, 6))
plt.plot(x, y)
plt.axis("equal")
plt.grid()
plt.title("Original Circle")
plt.show()


In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(x, y, s=10)
plt.axis("equal")
plt.grid()
plt.title("Sampled Points")

plt.show()

In [ ]:
# Close the curve by adding the first point at the end
x_closed = np.append(x, x[0])
y_closed = np.append(y, y[0])

# Distance between consecutive points
dx = np.diff(x_closed)
dy = np.diff(y_closed)
distances = np.sqrt(dx**2 + dy**2)

# Cumulative distance along the curve
distance = np.concatenate(([0], np.cumsum(distances)))

# Total length of the curve
total_length = distance[-1]

# New equally spaced distances
new_distance = np.linspace(0, total_length, N, endpoint=False)

# Interpolate x and y at the new distances
x_uniform = np.interp(new_distance, distance, x_closed)
y_uniform = np.interp(new_distance, distance, y_closed)

# Display the uniformly sampled points
plt.figure(figsize=(6, 6))
plt.scatter(x_uniform, y_uniform, s=10)
plt.axis("equal")
plt.grid()
plt.title("Approximately Uniformly Sampled Points")
plt.show()

In [ ]:
# Convert (x, y) points into complex numbers
z = x_uniform + 1j * y_uniform

# Display the first 10 complex points
print(z[:10])


In [ ]:
# Compute the Fourier Transform
fourier = np.fft.fft(z)

# Display the first 10 Fourier coefficients
print(fourier[:10])

In [ ]:
# Normalize the Fourier coefficients
coefficients = fourier / N

# Calculate the magnitude of each coefficient
magnitudes = np.abs(coefficients)

# Display the first 10 coefficients and their magnitudes
for i in range(10):
    print(i, coefficients[i], "magnitude =", magnitudes[i])

In [ ]:
# Frequencies corresponding to the FFT coefficients
frequencies = np.fft.fftfreq(N)

# Sort indices from largest magnitude to smallest
indices = np.argsort(magnitudes)[::-1]

# Sort everything using the same indices
sorted_coefficients = coefficients[indices]
sorted_frequencies = frequencies[indices]
sorted_magnitudes = magnitudes[indices]

# Display the first 10 sorted coefficients
for i in range(10):
    print(
        "Frequency:", sorted_frequencies[i],
        "Coefficient:", sorted_coefficients[i],
        "Magnitude:", sorted_magnitudes[i]
    )

In [ ]:
# Convert normalized FFT frequencies into integer frequencies
sorted_frequencies = frequencies[indices] * N

# Time values for reconstruction
t_reconstruct = np.arange(N) / N

# Create an empty array for the reconstructed curve
z_reconstructed = np.zeros(N, dtype=complex)

# Add each Fourier component
for i in range(N):
    z_reconstructed += (
        sorted_coefficients[i]
        * np.exp(2j * np.pi * sorted_frequencies[i] * t_reconstruct)
    )

# Plot original and reconstructed curves
plt.figure(figsize=(7, 7))

plt.plot(
    x_uniform,
    y_uniform,
    label="Original"
)

plt.plot(
    z_reconstructed.real,
    z_reconstructed.imag,
    "--",
    label="Reconstructed"
)

plt.axis("equal")
plt.grid()
plt.legend()
plt.title("Fourier Reconstruction")
plt.show()

In [ ]:
# Choose a time between 0 and 1
time = 0.25

# Starting point
current = 0 + 0j

# Store the points of the epicycle chain
points = [current]

# Use the largest 15 coefficients
num_epicycles = 15

for i in range(num_epicycles):
    coefficient = sorted_coefficients[i]
    frequency = sorted_frequencies[i]

    # Rotation at this time
    angle = 2 * np.pi * frequency * time

    # Rotating vector
    vector = coefficient * np.exp(1j * angle)

    # Add vector to current position
    current = current + vector

    points.append(current)

# Convert points to arrays
epicycle_x = np.array([p.real for p in points])
epicycle_y = np.array([p.imag for p in points])

# Draw the epicycle chain
plt.figure(figsize=(7, 7))

# Draw connecting vectors
plt.plot(epicycle_x, epicycle_y, "-o")

# Draw circles
for i in range(num_epicycles):
    center = points[i]
    radius = sorted_magnitudes[i]

    circle = plt.Circle(
        (center.real, center.imag),
        radius,
        fill=False
    )

    plt.gca().add_patch(circle)

plt.axis("equal")
plt.grid()
plt.title("Epicycle Chain")
plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Number of epicycles to display
num_epicycles = 15

# Number of animation frames
num_frames = 200

# Store the reconstructed path
path = []

# Create figure
fig, ax = plt.subplots(figsize=(7, 7))

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal")
ax.grid()

# Lines for vectors
vector_lines = []

# Circles
circles = []

for i in range(num_epicycles):
    line, = ax.plot([], [], "-o")
    vector_lines.append(line)

    circle = plt.Circle((0, 0), 0, fill=False)
    ax.add_patch(circle)
    circles.append(circle)

# Reconstructed path
path_line, = ax.plot([], [], linewidth=2)


def update(frame):

    time = frame / num_frames

    current = 0 + 0j
    points = [current]

    # Calculate epicycle positions
    for i in range(num_epicycles):

        coefficient = sorted_coefficients[i]
        frequency = sorted_frequencies[i]

        angle = 2 * np.pi * frequency * time

        vector = coefficient * np.exp(1j * angle)

        current = current + vector

        points.append(current)

    # Update vectors and circles
    for i in range(num_epicycles):

        x_values = [
            points[i].real,
            points[i + 1].real
        ]

        y_values = [
            points[i].imag,
            points[i + 1].imag
        ]

        vector_lines[i].set_data(x_values, y_values)

        circles[i].center = (
            points[i].real,
            points[i].imag
        )

        circles[i].radius = sorted_magnitudes[i]

    # Add endpoint to path
    path.append(points[-1])

    path_x = [p.real for p in path]
    path_y = [p.imag for p in path]

    path_line.set_data(path_x, path_y)

    return vector_lines + circles + [path_line]


animation = FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=30,
    blit=True
)

plt.close(fig)

HTML(animation.to_jshtml())


In [ ]:
def generate_curve(shape, N=500):

    t = np.linspace(0, 2*np.pi, N, endpoint=False)

    if shape == "Circle":
        x = np.cos(t)
        y = np.sin(t)

    elif shape == "Ellipse":
        x = 1.5 * np.cos(t)
        y = 0.8 * np.sin(t)

    elif shape == "Heart":
        x = 16 * np.sin(t)**3
        y = (
            13 * np.cos(t)
            - 5 * np.cos(2*t)
            - 2 * np.cos(3*t)
            - np.cos(4*t)
        )

        # Scale the heart
        x = x / 18
        y = y / 18

    elif shape == "Star":
        r = 1 + 0.4 * np.cos(5*t)
        x = r * np.cos(t)
        y = r * np.sin(t)

    elif shape == "Oval":
        x = np.cos(t)
        y = 0.6 * np.sin(t)

    elif shape == "Egg":
        x = np.cos(t)
        y = 0.8 * np.sin(t) + 0.2 * np.sin(t)**2

    return x, y


# Select a shape
shape = "Circle"

# Generate the curve
x, y = generate_curve(shape)

# Display it
plt.figure(figsize=(6, 6))
plt.plot(x, y)
plt.scatter(x, y, s=3)

plt.axis("equal")
plt.grid()
plt.title(shape)

plt.show()

In [ ]:
# Number of Fourier samples
N = 500

# Generate the selected curve
x, y = generate_curve(shape, N)

# Close the curve
x_closed = np.append(x, x[0])
y_closed = np.append(y, y[0])

# Calculate distance between consecutive points
dx = np.diff(x_closed)
dy = np.diff(y_closed)

distances = np.sqrt(dx**2 + dy**2)

# Cumulative distance
distance = np.concatenate(([0], np.cumsum(distances)))

# Total curve length
total_length = distance[-1]

# Uniform distances
new_distance = np.linspace(0, total_length, N, endpoint=False)

# Uniformly sampled points
x_uniform = np.interp(new_distance, distance, x_closed)
y_uniform = np.interp(new_distance, distance, y_closed)

# Convert to complex numbers
z = x_uniform + 1j * y_uniform

# Fourier Transform
fourier = np.fft.fft(z)

# Normalize
coefficients = fourier / N

# Fourier frequencies
frequencies = np.fft.fftfreq(N) * N

# Magnitudes
magnitudes = np.abs(coefficients)

# Sort by magnitude
indices = np.argsort(magnitudes)[::-1]

sorted_coefficients = coefficients[indices]
sorted_frequencies = frequencies[indices]
sorted_magnitudes = magnitudes[indices]

print("Shape:", shape)
print("Number of samples:", N)
print("Curve length:", total_length)
print()
print("Largest 10 Fourier magnitudes:")

for i in range(10):
    print(
        i,
        "Frequency =", sorted_frequencies[i],
        "Magnitude =", sorted_magnitudes[i]
    )

In [ ]:
# Time values
t_reconstruct = np.arange(N) / N

# Number of coefficients to use
num_coefficients = 5

# Start with zero
z_reconstructed = np.zeros(N, dtype=complex)

# Add Fourier components
for i in range(num_coefficients):

    coefficient = sorted_coefficients[i]
    frequency = sorted_frequencies[i]

    z_reconstructed += (
        coefficient
        * np.exp(2j * np.pi * frequency * t_reconstruct)
    )

# Plot original and reconstructed circle
plt.figure(figsize=(7, 7))

plt.plot(
    x_uniform,
    y_uniform,
    label="Original Circle"
)

plt.plot(
    z_reconstructed.real,
    z_reconstructed.imag,
    "--",
    label="Reconstructed Circle"
)

plt.axis("equal")
plt.grid()
plt.legend()
plt.title("Circle Reconstruction")

plt.show()


In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Number of coefficients to display
num_epicycles = 5

# Number of animation frames
num_frames = 200

# Store the path
path = []

# Create figure
fig, ax = plt.subplots(figsize=(7, 7))

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal")
ax.grid()

# Original circle
ax.plot(
    x_uniform,
    y_uniform,
    "--",
    alpha=0.3
)

# Lines for epicycle vectors
vector_lines = []

# Circles
circles = []

for i in range(num_epicycles):

    line, = ax.plot([], [], "-o")
    vector_lines.append(line)

    circle = plt.Circle(
        (0, 0),
        0,
        fill=False
    )

    ax.add_patch(circle)
    circles.append(circle)

# Reconstructed path
path_line, = ax.plot([], [], linewidth=2)


def update(frame):

    time = frame / num_frames

    current = 0 + 0j
    points = [current]

    # Calculate epicycle chain
    for i in range(num_epicycles):

        coefficient = sorted_coefficients[i]
        frequency = sorted_frequencies[i]

        angle = 2 * np.pi * frequency * time

        vector = coefficient * np.exp(1j * angle)

        current = current + vector

        points.append(current)

    # Update each vector and circle
    for i in range(num_epicycles):

        vector_lines[i].set_data(
            [points[i].real, points[i + 1].real],
            [points[i].imag, points[i + 1].imag]
        )

        circles[i].center = (
            points[i].real,
            points[i].imag
        )

        circles[i].radius = sorted_magnitudes[i]

    # Add the endpoint to the path
    path.append(points[-1])

    path_line.set_data(
        [p.real for p in path],
        [p.imag for p in path]
    )

    return vector_lines + circles + [path_line]


animation = FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=30,
    blit=True
)

plt.close(fig)

HTML(animation.to_jshtml())

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create dropdown for selecting the curve

shape_dropdown = widgets.Dropdown(
    options=[
        "Circle",
        "Ellipse",
        "Heart",
        "Star",
        "Oval",
        "Egg"
    ],
    value="Circle",
    description="Curve:"
)

display(shape_dropdown)

In [ ]:
# Get the selected curve
shape = shape_dropdown.value

# Number of samples
N = 500

# Generate the curve
x, y = generate_curve(shape, N)

# Close the curve
x_closed = np.append(x, x[0])
y_closed = np.append(y, y[0])

# Calculate distances between points
dx = np.diff(x_closed)
dy = np.diff(y_closed)

distances = np.sqrt(dx**2 + dy**2)

# Cumulative arc length
distance = np.concatenate(([0], np.cumsum(distances)))

# Total curve length
total_length = distance[-1]

# Uniform sampling
new_distance = np.linspace(
    0,
    total_length,
    N,
    endpoint=False
)

x_uniform = np.interp(
    new_distance,
    distance,
    x_closed
)

y_uniform = np.interp(
    new_distance,
    distance,
    y_closed
)

# Convert to complex numbers
z = x_uniform + 1j * y_uniform

# Fourier Transform
fourier = np.fft.fft(z)

# Normalize
coefficients = fourier / N

# Frequencies
frequencies = np.fft.fftfreq(N) * N

# Magnitudes
magnitudes = np.abs(coefficients)

# Sort by magnitude
indices = np.argsort(magnitudes)[::-1]

sorted_coefficients = coefficients[indices]
sorted_frequencies = frequencies[indices]
sorted_magnitudes = magnitudes[indices]

print("Selected curve:", shape)
print("Number of samples:", N)
print("Total curve length:", total_length)
print("Fourier coefficients calculated:", len(coefficients))

In [ ]:
def get_epicycles(t, number_of_epicycles=20):

    current = 0 + 0j

    points = [current]

    for i in range(number_of_epicycles):

        coefficient = sorted_coefficients[i]
        frequency = sorted_frequencies[i]

        angle = 2 * np.pi * frequency * t

        vector = coefficient * np.exp(1j * angle)

        current = current + vector

        points.append(current)

    return points

    points = get_epicycles(0.25, 20)



In [ ]:
# Choose a time
t = 0.25

# Get epicycle positions
points = get_epicycles(t, 20)

# Create figure
plt.figure(figsize=(7, 7))

# Draw the connecting vectors
for i in range(len(points) - 1):

    plt.plot(
        [points[i].real, points[i + 1].real],
        [points[i].imag, points[i + 1].imag],
        "-o"
    )

# Draw the epicycle circles
for i in range(len(points) - 1):

    center = points[i]
    radius = sorted_magnitudes[i]

    circle = plt.Circle(
        (center.real, center.imag),
        radius,
        fill=False
    )

    plt.gca().add_patch(circle)

# Draw the reconstructed endpoint
plt.scatter(
    points[-1].real,
    points[-1].imag,
    s=40
)

plt.axis("equal")
plt.grid()
plt.title("Epicycle Chain at t = 0.25")

plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Number of epicycles
num_epicycles = 20

# Number of animation frames
num_frames = 200

# Store the reconstructed path
path = []

# Create figure
fig, ax = plt.subplots(figsize=(7, 7))

ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect("equal")
ax.grid()

# Original curve
ax.plot(
    x_uniform,
    y_uniform,
    "--",
    alpha=0.3
)

# Create vector lines
vector_lines = []

# Create circles
circles = []

for i in range(num_epicycles):

    line, = ax.plot([], [], "-o")
    vector_lines.append(line)

    circle = plt.Circle(
        (0, 0),
        0,
        fill=False
    )

    ax.add_patch(circle)
    circles.append(circle)

# Reconstructed path
path_line, = ax.plot([], [], linewidth=2)


def update(frame):

    t = frame / num_frames

    # Get the epicycle positions
    points = get_epicycles(
        t,
        num_epicycles
    )

    # Update vectors and circles
    for i in range(num_epicycles):

        vector_lines[i].set_data(
            [points[i].real, points[i + 1].real],
            [points[i].imag, points[i + 1].imag]
        )

        circles[i].center = (
            points[i].real,
            points[i].imag
        )

        circles[i].radius = sorted_magnitudes[i]

    # Add final endpoint to path
    path.append(points[-1])

    path_line.set_data(
        [p.real for p in path],
        [p.imag for p in path]
    )

    return vector_lines + circles + [path_line]


animation = FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=30,
    blit=True
)

plt.close(fig)

HTML(animation.to_jshtml())

In [ ]:
def process_curve(shape, N=500):

    # Generate curve
    x, y = generate_curve(shape, N)

    # Close curve
    x_closed = np.append(x, x[0])
    y_closed = np.append(y, y[0])

    # Calculate distances
    dx = np.diff(x_closed)
    dy = np.diff(y_closed)

    distances = np.sqrt(dx**2 + dy**2)

    # Cumulative arc length
    distance = np.concatenate(([0], np.cumsum(distances)))

    # Total curve length
    total_length = distance[-1]

    # Uniform sampling
    new_distance = np.linspace(
        0,
        total_length,
        N,
        endpoint=False
    )

    x_uniform = np.interp(
        new_distance,
        distance,
        x_closed
    )

    y_uniform = np.interp(
        new_distance,
        distance,
        y_closed
    )

    # Convert to complex numbers
    z = x_uniform + 1j * y_uniform

    # Fourier Transform
    fourier = np.fft.fft(z)

    # Normalize
    coefficients = fourier / N

    # Frequencies
    frequencies = np.fft.fftfreq(N) * N

    # Magnitudes
    magnitudes = np.abs(coefficients)

    # Sort by magnitude
    indices = np.argsort(magnitudes)[::-1]

    sorted_coefficients = coefficients[indices]
    sorted_frequencies = frequencies[indices]
    sorted_magnitudes = magnitudes[indices]

    return (
        x_uniform,
        y_uniform,
        sorted_coefficients,
        sorted_frequencies,
        sorted_magnitudes
    )


# Test the function with a circle
x_uniform, y_uniform, sorted_coefficients, sorted_frequencies, sorted_magnitudes = process_curve(
    "Circle"
)

print("Circle processed successfully")
print("Number of Fourier coefficients:", len(sorted_coefficients))

In [ ]:
# Get the curve selected in the dropdown
shape = shape_dropdown.value

# Process the selected curve
x_uniform, y_uniform, sorted_coefficients, sorted_frequencies, sorted_magnitudes = process_curve(
    shape
)

# Display the selected curve
plt.figure(figsize=(6, 6))

plt.plot(
    x_uniform,
    y_uniform
)

plt.axis("equal")
plt.grid()
plt.title("Selected Curve: " + shape)

plt.show()

print("Processed:", shape)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Number of epicycles
num_epicycles = 20

# Number of animation frames
num_frames = 200

# Store the reconstructed path
path = []

# Create figure
fig, ax = plt.subplots(figsize=(7, 7))

# Find a suitable plot size
limit = max(
    np.max(np.abs(x_uniform)),
    np.max(np.abs(y_uniform))
) * 1.5

ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)

ax.set_aspect("equal")
ax.grid()

# Original curve
ax.plot(
    x_uniform,
    y_uniform,
    "--",
    alpha=0.3
)

# Create epicycle vector lines
vector_lines = []

# Create epicycle circles
circles = []

for i in range(num_epicycles):

    line, = ax.plot([], [], "-o")
    vector_lines.append(line)

    circle = plt.Circle(
        (0, 0),
        0,
        fill=False
    )

    ax.add_patch(circle)
    circles.append(circle)

# Reconstructed path
path_line, = ax.plot([], [], linewidth=2)


def update(frame):

    t = frame / num_frames

    # Calculate epicycle positions
    points = get_epicycles(
        t,
        num_epicycles
    )

    # Update vectors and circles
    for i in range(num_epicycles):

        vector_lines[i].set_data(
            [points[i].real, points[i + 1].real],
            [points[i].imag, points[i + 1].imag]
        )

        circles[i].center = (
            points[i].real,
            points[i].imag
        )

        circles[i].radius = sorted_magnitudes[i]

    # Add endpoint to reconstructed path
    path.append(points[-1])

    path_line.set_data(
        [p.real for p in path],
        [p.imag for p in path]
    )

    return vector_lines + circles + [path_line]


animation = FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=30,
    blit=True
)

plt.close(fig)

HTML(animation.to_jshtml())

In [ ]:
from google.colab.output import eval_js
from IPython.display import display, HTML

display(HTML("""
<div style="text-align:center;">
    <canvas id="drawCanvas"
            width="600"
            height="500"
            style="border:2px solid black; cursor:crosshair;">
    </canvas>

    <br><br>

    <button onclick="clearCanvas()">
        Clear
    </button>

    <button onclick="finishDrawing()">
        Done Drawing
    </button>
</div>

<script>

var canvas = document.getElementById("drawCanvas");
var ctx = canvas.getContext("2d");

var drawing = false;

ctx.lineWidth = 2;
ctx.lineCap = "round";

var points = [];

canvas.addEventListener("mousedown", function(event) {

    drawing = true;

    var rect = canvas.getBoundingClientRect();

    var x = event.clientX - rect.left;
    var y = event.clientY - rect.top;

    points.push([x, y]);

    ctx.beginPath();
    ctx.moveTo(x, y);
});


canvas.addEventListener("mousemove", function(event) {

    if (!drawing) {
        return;
    }

    var rect = canvas.getBoundingClientRect();

    var x = event.clientX - rect.left;
    var y = event.clientY - rect.top;

    points.push([x, y]);

    ctx.lineTo(x, y);
    ctx.stroke();
});


canvas.addEventListener("mouseup", function() {
    drawing = false;
});


function clearCanvas() {

    ctx.clearRect(
        0,
        0,
        canvas.width,
        canvas.height
    );

    points = [];
}


function finishDrawing() {

    google.colab.kernel.invokeFunction(
        'notebook.get_points',
        [points],
        {}
    );
}

</script>
"""))

In [ ]:
from google.colab import output

def automatic_process(points):

    global drawn_points
    global draw_x
    global draw_y
    global draw_x_uniform
    global draw_y_uniform
    global sorted_coefficients_draw
    global sorted_frequencies_draw
    global sorted_magnitudes_draw
    global drawing_animation

    # Convert points to NumPy array
    drawn_points = np.array(points)

    print("Drawn points:", len(drawn_points))

    # Separate x and y
    draw_x = drawn_points[:, 0]
    draw_y = drawn_points[:, 1]

    # Number of Fourier samples
    N = 500

    # Close the curve
    x_closed = np.append(draw_x, draw_x[0])
    y_closed = np.append(draw_y, draw_y[0])

    # Calculate distances
    dx = np.diff(x_closed)
    dy = np.diff(y_closed)

    distances = np.sqrt(dx**2 + dy**2)

    # Cumulative distance
    distance = np.concatenate(([0], np.cumsum(distances)))

    # Total length
    total_length = distance[-1]

    # Uniform sampling
    new_distance = np.linspace(
        0,
        total_length,
        N,
        endpoint=False
    )

    draw_x_uniform = np.interp(
        new_distance,
        distance,
        x_closed
    )

    draw_y_uniform = np.interp(
        new_distance,
        distance,
        y_closed
    )

    # Convert to complex numbers
    z_draw = draw_x_uniform + 1j * draw_y_uniform

    # Fourier Transform
    fourier_draw = np.fft.fft(z_draw)

    # Normalize
    coefficients_draw = fourier_draw / N

    # Frequencies
    frequencies_draw = np.fft.fftfreq(N) * N

    # Magnitudes
    magnitudes_draw = np.abs(coefficients_draw)

    # Sort by magnitude
    indices_draw = np.argsort(magnitudes_draw)[::-1]

    sorted_coefficients_draw = coefficients_draw[indices_draw]
    sorted_frequencies_draw = frequencies_draw[indices_draw]
    sorted_magnitudes_draw = magnitudes_draw[indices_draw]

    print("Fourier processing complete.")
    print("Reconstruction data ready.")

    # Create animation
    drawing_animation = create_drawing_animation()

    print("Animation created successfully.")


output.register_callback(
    "notebook.get_points",
    automatic_process
)

In [ ]:
def create_drawing_animation():

    # Number of epicycles
    num_epicycles = 50

    # Number of animation frames
    num_frames = 300

    # Create figure
    fig, ax = plt.subplots(figsize=(8, 6))

    # Find suitable plot limits
    limit_x = np.max(np.abs(draw_x_uniform))
    limit_y = np.max(np.abs(draw_y_uniform))

    limit = max(limit_x, limit_y) * 1.2

    ax.set_xlim(-limit, limit)
    ax.set_ylim(-limit, limit)

    ax.set_aspect("equal")
    ax.grid()

    # Original drawing
    ax.plot(
        draw_x_uniform,
        -draw_y_uniform,
        "--",
        alpha=0.3
    )

    # Vector lines
    vector_lines = []

    # Epicycle circles
    circles = []

    for i in range(num_epicycles):

        line, = ax.plot([], [], "-o")
        vector_lines.append(line)

        circle = plt.Circle(
            (0, 0),
            0,
            fill=False
        )

        ax.add_patch(circle)
        circles.append(circle)

    # Reconstructed path
    path_line, = ax.plot(
        [],
        [],
        linewidth=2
    )

    # Store path
    path = []

    def update(frame):

        t = frame / num_frames

        current = 0 + 0j

        points = [current]

        # Calculate epicycles
        for i in range(num_epicycles):

            coefficient = sorted_coefficients_draw[i]
            frequency = sorted_frequencies_draw[i]

            angle = 2 * np.pi * frequency * t

            vector = coefficient * np.exp(1j * angle)

            current = current + vector

            points.append(current)

        # Update vectors and circles
        for i in range(num_epicycles):

            vector_lines[i].set_data(
                [points[i].real, points[i + 1].real],
                [-points[i].imag, -points[i + 1].imag]
            )

            circles[i].center = (
                points[i].real,
                -points[i].imag
            )

            circles[i].radius = sorted_magnitudes_draw[i]

        # Add endpoint to path
        path.append(points[-1])

        path_line.set_data(
            [p.real for p in path],
            [-p.imag for p in path]
        )

        return vector_lines + circles + [path_line]

    # Create animation
    animation = FuncAnimation(
        fig,
        update,
        frames=num_frames,
        interval=30,
        blit=True
    )

    plt.close(fig)

    return HTML(animation.to_jshtml())

In [ ]:
# Select a curve from the dropdown
shape = shape_dropdown.value

# Process the selected curve
x_uniform, y_uniform, sorted_coefficients, sorted_frequencies, sorted_magnitudes = process_curve(
    shape
)

print("Selected curve:", shape)
print("Fourier processing complete.")

# Start the epicycle animation
num_epicycles = 50
num_frames = 300
path = []

fig, ax = plt.subplots(figsize=(7, 7))

limit = max(
    np.max(np.abs(x_uniform)),
    np.max(np.abs(y_uniform))
) * 1.3

ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)
ax.set_aspect("equal")
ax.grid()

# Original curve
ax.plot(
    x_uniform,
    y_uniform,
    "--",
    alpha=0.3
)

vector_lines = []
circles = []

for i in range(num_epicycles):

    line, = ax.plot([], [], "-o")
    vector_lines.append(line)

    circle = plt.Circle(
        (0, 0),
        0,
        fill=False
    )

    ax.add_patch(circle)
    circles.append(circle)

path_line, = ax.plot([], [], linewidth=2)


def update_predefined(frame):

    t = frame / num_frames

    points = get_epicycles(
        t,
        num_epicycles
    )

    for i in range(num_epicycles):

        vector_lines[i].set_data(
            [points[i].real, points[i + 1].real],
            [points[i].imag, points[i + 1].imag]
        )

        circles[i].center = (
            points[i].real,
            points[i].imag
        )

        circles[i].radius = sorted_magnitudes[i]

    path.append(points[-1])

    path_line.set_data(
        [p.real for p in path],
        [p.imag for p in path]
    )

    return vector_lines + circles + [path_line]


animation = FuncAnimation(
    fig,
    update_predefined,
    frames=num_frames,
    interval=30,
    blit=True
)

plt.close(fig)

HTML(animation.to_jshtml())

In [ ]:
# Store the points drawn with the mouse

drawn_points = []


def receive_points(points):

    global drawn_points

    drawn_points = np.array(points)

    print("Number of drawn points:", len(drawn_points))


# Register the Python function so JavaScript can call it
from google.colab import output

output.register_callback(
    "notebook.get_points",
    receive_points
)

In [ ]:
# Separate x and y coordinates

draw_x = drawn_points[:, 0]
draw_y = drawn_points[:, 1]

# Plot the drawing
plt.figure(figsize=(7, 6))

plt.plot(
    draw_x,
    -draw_y
)

plt.axis("equal")
plt.grid()
plt.title("Your Mouse Drawing")

plt.show()

In [ ]:
# Number of points for Fourier analysis
N = 500

# Close the drawn curve
x_closed = np.append(draw_x, draw_x[0])
y_closed = np.append(draw_y, draw_y[0])

# Calculate distance between consecutive points
dx = np.diff(x_closed)
dy = np.diff(y_closed)

distances = np.sqrt(dx**2 + dy**2)

# Cumulative distance along the curve
distance = np.concatenate(([0], np.cumsum(distances)))

# Total length of the drawn curve
total_length = distance[-1]

# Create equally spaced distances
new_distance = np.linspace(
    0,
    total_length,
    N,
    endpoint=False
)

# Interpolate new points
draw_x_uniform = np.interp(
    new_distance,
    distance,
    x_closed
)

draw_y_uniform = np.interp(
    new_distance,
    distance,
    y_closed
)

# Display the uniformly sampled drawing
plt.figure(figsize=(7, 6))

plt.plot(
    draw_x_uniform,
    -draw_y_uniform
)

plt.axis("equal")
plt.grid()
plt.title("Uniformly Sampled Drawing")

plt.show()

print("Original drawn points:", len(drawn_points))
print("Uniformly sampled points:", N)
print("Approximate curve length:", total_length)

In [ ]:
# Convert the uniformly sampled drawing to complex numbers

z_draw = draw_x_uniform + 1j * draw_y_uniform

# Fourier Transform
fourier_draw = np.fft.fft(z_draw)

# Normalize the Fourier coefficients
coefficients_draw = fourier_draw / N

# Frequencies
frequencies_draw = np.fft.fftfreq(N) * N

# Magnitudes
magnitudes_draw = np.abs(coefficients_draw)

# Sort by magnitude
indices_draw = np.argsort(magnitudes_draw)[::-1]

sorted_coefficients_draw = coefficients_draw[indices_draw]
sorted_frequencies_draw = frequencies_draw[indices_draw]
sorted_magnitudes_draw = magnitudes_draw[indices_draw]

print("Fourier coefficients:", len(sorted_coefficients_draw))
print()
print("Largest 10 Fourier components:")

for i in range(10):
    print(
        i,
        "Frequency =", sorted_frequencies_draw[i],
        "Magnitude =", sorted_magnitudes_draw[i]
    )

In [ ]:
# Number of Fourier coefficients to use
num_coefficients = 500

# Time values
t_reconstruct = np.arange(N) / N

# Start with zero
z_draw_reconstructed = np.zeros(N, dtype=complex)

# Add Fourier components
for i in range(num_coefficients):

    coefficient = sorted_coefficients_draw[i]
    frequency = sorted_frequencies_draw[i]

    z_draw_reconstructed += (
        coefficient
        * np.exp(2j * np.pi * frequency * t_reconstruct)
    )

# Plot original drawing and reconstruction
plt.figure(figsize=(8, 6))

plt.plot(
    draw_x_uniform,
    -draw_y_uniform,
    label="Original Drawing"
)

plt.plot(
    z_draw_reconstructed.real,
    -z_draw_reconstructed.imag,
    "--",
    label="Fourier Reconstruction"
)

plt.axis("equal")
plt.grid()
plt.legend()

plt.title(
    "Mouse Drawing Reconstruction - "
    + str(num_coefficients)
    + " Coefficients"
)

plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Number of epicycles
num_epicycles = 50

# Number of animation frames
num_frames = 300

# Store reconstructed path
path = []

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Find suitable plot limits
limit_x = np.max(np.abs(draw_x_uniform))
limit_y = np.max(np.abs(draw_y_uniform))

limit = max(limit_x, limit_y) * 1.2

ax.set_xlim(-limit, limit)
ax.set_ylim(-limit, limit)

ax.set_aspect("equal")
ax.grid()

# Original drawing
ax.plot(
    draw_x_uniform,
    -draw_y_uniform,
    "--",
    alpha=0.3,
    label="Original Drawing"
)

# Vector lines
vector_lines = []

# Epicycle circles
circles = []

for i in range(num_epicycles):

    line, = ax.plot([], [], "-o")
    vector_lines.append(line)

    circle = plt.Circle(
        (0, 0),
        0,
        fill=False
    )

    ax.add_patch(circle)
    circles.append(circle)

# Reconstructed path
path_line, = ax.plot(
    [],
    [],
    linewidth=2,
    label="Reconstruction"
)


def update(frame):

    # Current time
    t = frame / num_frames

    # Starting point
    current = 0 + 0j

    # Store epicycle points
    points = [current]

    # Calculate epicycles
    for i in range(num_epicycles):

        coefficient = sorted_coefficients_draw[i]
        frequency = sorted_frequencies_draw[i]

        angle = 2 * np.pi * frequency * t

        vector = coefficient * np.exp(1j * angle)

        current = current + vector

        points.append(current)

    # Update vectors and circles
    for i in range(num_epicycles):

        vector_lines[i].set_data(
            [points[i].real, points[i + 1].real],
            [-points[i].imag, -points[i + 1].imag]
        )

        circles[i].center = (
            points[i].real,
            -points[i].imag
        )

        circles[i].radius = sorted_magnitudes_draw[i]

    # Add final endpoint to path
    path.append(points[-1])

    path_line.set_data(
        [p.real for p in path],
        [-p.imag for p in path]
    )

    return vector_lines + circles + [path_line]


animation = FuncAnimation(
    fig,
    update,
    frames=num_frames,
    interval=30,
    blit=True
)

plt.close(fig)

HTML(animation.to_jshtml())